# 27 Parafrase dan Augmentasi

Menambah data latih dengan teks baru, lalu diukur apakah benar-benar menaikkan skor. Contoh: 500 baris SmSA berlabel.

**Daftar isi**

1. Kartu Masalah
2. Kapan Pakai Notebook Ini
3. Config
4. EDA
5. Preprocessing
6. Validasi
7. Baseline
8. Model Utama
9. Post-processing
10. Post-analysis
11. Keluaran

## 1. Kartu Masalah

| Bagian | Isi |
|---|---|
| **Task** | Menambah data dengan parafrase dan augmentasi |
| **Unit prediksi** | Satu teks latih mendapat salinan yang diubah |
| **Metrik** | Macro-F1, dan kemiripan makna antara teks asli dan salinan |
| **Masalah terbesar** | Holdout bergeser dan label sedikit (500 baris): OOF tanpa augmentasi 0,8336, tapi holdout hanya 0,6265. Augmentasi yang salah tempat juga membuat skor palsu. |
| **Data contoh** | SmSA: 500 latih, 500 holdout |

## 2. Kapan Pakai Notebook Ini

**Pakai kalau** data berlabel sedikit, atau soal meminta membuat variasi teks. Kalau ada banyak data tanpa label,
coba folder `08` dulu, karena pseudo-label memakai teks asli.

**Yang perlu diganti kalau datanya beda:**
- `load_data`, `CLASSES`.
- Jenis augmentasi di `AUGMENTERS`. Hapus yang label-nya sering berubah.

## 3. Config

**Import dan lingkungan.** Mode offline dinyalakan di sini, dan progress bar serta warning
dimatikan supaya output tetap bersih.

In [1]:
import importlib.util
import json
import os
import random
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

os.environ.update(
    CUDA_VISIBLE_DEVICES="0",
    HF_HUB_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
    TOKENIZERS_PARALLELISM="false",
    TQDM_DISABLE="1",
    PYTHONWARNINGS="ignore",
    HF_HUB_DISABLE_PROGRESS_BARS="1",
)
warnings.filterwarnings("ignore")
sys.modules["tensorflow"] = None  # tidak dipakai; mencegah log TensorFlow saat umap diimpor

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90, "display.max_rows", 30, "display.width", 140)
plt.rcParams.update(
    {
        "figure.figsize": (8, 3.5),
        "figure.dpi": 90,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(f"device={DEVICE} | torch={torch.__version__}")

device=cuda | torch=2.11.0+cu130


**Lokasi file.** `find_pack` mencari paket offline, dan `find_file` mencari file data di Kaggle
atau folder lokal. Di mesin lokal, set `PACK_DIR` dan `DATA_ROOT` lewat environment variable.

In [2]:
INPUT_ROOTS = [Path(p) for p in (os.environ.get("DATA_ROOT"), "/kaggle/input", ".") if p]


def find_pack() -> Path:
    """Cari folder paket offline lewat file pack_manifest.json."""
    if os.environ.get("PACK_DIR"):
        return Path(os.environ["PACK_DIR"])
    for root in INPUT_ROOTS:
        for manifest in root.glob("**/pack_manifest.json"):
            return manifest.parent
    raise FileNotFoundError("Paket offline tidak ketemu. Set environment variable PACK_DIR.")


def find_file(name: str) -> Path:
    """Cari file data berdasarkan nama di semua folder input."""
    for root in INPUT_ROOTS:
        for path in root.glob(f"**/{name}"):
            return path
    raise FileNotFoundError(f"File {name} tidak ketemu di {INPUT_ROOTS}")


def install_offline(*modules: str) -> None:
    """Pasang library dari wheel paket offline, hanya bila belum terpasang."""
    missing = [m for m in modules if importlib.util.find_spec(m.replace("-", "_")) is None]
    if missing:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-index",
                "--find-links",
                str(PACK_DIR / "wheels"),
                "--root-user-action=ignore",
                *missing,
            ],
            check=True,
        )


PACK_DIR = find_pack()
MODEL_DIR = PACK_DIR / "models"
EXTRA_DATA_DIR = PACK_DIR / "data"
RESULT_DIR = Path("hasil")
RESULT_DIR.mkdir(exist_ok=True)
print(f"paket offline: {PACK_DIR}")

paket offline: /kaggle/input/notebooks/ivanwllm/gemastik-nlp-00-setup-offline


**Pencatat waktu dan ringkasan.** `stage` mencatat durasi tiap tahap, dan `run_summary` mencetak
ringkasan akhir yang dibaca skrip audit.

In [3]:
RUN_START = time.time()
STAGE_SECONDS: dict[str, float] = {}


@contextmanager
def stage(name: str):
    """Catat durasi satu tahap."""
    start = time.time()
    yield
    STAGE_SECONDS[name] = round(time.time() - start, 1)
    print(f"[waktu] {name}: {STAGE_SECONDS[name]:.0f} detik")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_summary(task: str, n_rows: int, metrics: dict, needs_gpu: bool = True) -> None:
    """Simpan dan cetak ringkasan run."""
    total = int(time.time() - RUN_START)
    used_gpu = torch.cuda.is_available() and torch.cuda.max_memory_allocated() > 0
    summary = {
        "task": task,
        "mode": MODE,
        "use_pretrained": USE_PRETRAINED,
        "n_rows": n_rows,
        "total_detik": total,
        "gpu": used_gpu,
        "tahap_detik": STAGE_SECONDS,
        "metrik": metrics,
    }
    (RESULT_DIR / "ringkasan.json").write_text(json.dumps(summary, indent=1))
    note = "" if needs_gpu else " gpu_tidak_wajib"
    print(
        f"RINGKASAN_RUN task={task} mode={MODE} n_rows={n_rows} total_detik={total} "
        f"gpu={used_gpu}{note}"
    )
    print(json.dumps(metrics, indent=1))

**Pengaturan utama.** Semua yang perlu diubah ada di sel ini.

In [4]:
MODE = "penuh"  # "cepat" (<30 menit) atau "penuh" (<2 jam)
USE_PRETRAINED = True  # False: hanya augmentasi kata, tanpa file model dari luar
SEED = 42
N_LABELED = 500
CLASSES = ["negative", "neutral", "positive"]

SETTINGS = {
    "cepat": {"folds": 3, "epochs": 4, "seeds": [SEED]},
    "penuh": {"folds": 5, "epochs": 6, "seeds": [SEED, SEED + 1]},
}[MODE]

seed_everything(SEED)
print(SETTINGS)

{'folds': 5, 'epochs': 6, 'seeds': [42, 43]}


**Util teks.** `clean_text` hanya membuang gangguan teknis (URL, mention, spasi) dan
mempertahankan emoji serta tanda seru karena keduanya membawa sinyal. `normalize_slang` memakai
kamus alay. `near_duplicate_groups` menemukan komentar kembar supaya tidak bocor antar fold.

In [5]:
import re  # noqa: E402

from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import roc_auc_score  # noqa: E402
from sklearn.model_selection import cross_val_predict  # noqa: E402
from sklearn.neighbors import NearestNeighbors  # noqa: E402

URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
SPACE_RE = re.compile(r"\s+")


def clean_text(text: str) -> str:
    """Bersihkan URL, mention, dan spasi berlebih. Huruf kecil, emoji tetap."""
    text = URL_RE.sub(" ", str(text))
    text = MENTION_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip().lower()


def load_slang_dict() -> dict[str, str]:
    """Muat kamus kata informal ke kata baku."""
    path = next((EXTRA_DATA_DIR / "kamus_alay").glob("**/colloquial-indonesian-lexicon.csv"))
    lexicon = pd.read_csv(path)
    return dict(zip(lexicon["slang"].astype(str), lexicon["formal"].astype(str)))


def normalize_slang(text: str, slang: dict[str, str]) -> str:
    """Ganti kata informal dengan kata baku, per kata."""
    return " ".join(slang.get(word, word) for word in text.split())


def near_duplicate_groups(texts: list[str], threshold: float = 0.9) -> np.ndarray:
    """Beri id grup yang sama pada teks yang kemiripan char TF-IDF-nya di atas threshold."""
    vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    neighbors = NearestNeighbors(n_neighbors=min(6, len(texts)), metric="cosine").fit(vectors)
    distance, index = neighbors.kneighbors(vectors)
    parent = list(range(len(texts)))

    def root(i: int) -> int:
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, (row_dist, row_idx) in enumerate(zip(distance, index)):
        for dist, j in zip(row_dist, row_idx):
            if dist <= 1 - threshold:
                parent[root(i)] = root(int(j))
    return np.array([root(i) for i in range(len(texts))])


def adversarial_auc(texts_a: list[str], texts_b: list[str]) -> float:
    """AUC model yang membedakan dua kumpulan teks. Dekat 0,5 berarti tidak ada shift."""
    texts = list(texts_a) + list(texts_b)
    target = np.array([0] * len(texts_a) + [1] * len(texts_b))
    vectors = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scores = cross_val_predict(model, vectors, target, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(target, scores))


def text_profile(texts: pd.Series) -> pd.Series:
    """Ringkasan kualitas teks untuk EDA."""
    words = texts.astype(str).str.split().str.len()
    return pd.Series(
        {
            "baris": len(texts),
            "kosong": int((words == 0).sum()),
            "duplikat_persis_%": round(100 * texts.duplicated().mean(), 1),
            "kata_median": float(words.median()),
            "kata_p95": float(words.quantile(0.95)),
            "kata_maks": int(words.max()),
            "ada_url_%": round(100 * texts.astype(str).str.contains(r"https?://").mean(), 1),
            "ada_emoji_%": round(
                100 * texts.astype(str).str.contains(r"[\U0001F300-\U0001FAFF]").mean(), 1
            ),
        }
    )

**Metrik, interval noise, dan pencatat hasil.** Metrik diuji dulu pada contoh yang jawabannya
dihitung manual. `record` mencatat skor OOF dan skor holdout untuk tiap pendekatan.

In [6]:
from sklearn.metrics import f1_score  # noqa: E402
from sklearn.model_selection import StratifiedGroupKFold  # noqa: E402


def macro_f1(y_true, y_pred) -> float:
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))


assert abs(macro_f1([0, 0, 1, 1], [0, 1, 1, 1]) - (2 / 3 + 0.8) / 2) < 1e-9


def bootstrap_interval(y_true, y_pred, score_fn=macro_f1, n_boot: int = 300) -> str:
    """Interval 95%. Selisih di dalam interval ini belum tentu perbaikan nyata."""
    rng = np.random.default_rng(SEED)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(y_true), len(y_true))
        scores.append(score_fn(y_true[pick], y_pred[pick]))
    return f"{np.percentile(scores, 2.5):.3f}-{np.percentile(scores, 97.5):.3f}"


def make_folds(target, groups, n_folds: int) -> np.ndarray:
    """StratifiedGroupKFold: duplikat tidak lintas fold, proporsi kelas terjaga."""
    splitter = StratifiedGroupKFold(n_folds, shuffle=True, random_state=SEED)
    folds = np.zeros(len(target), dtype=int)
    for fold, (_, val_idx) in enumerate(splitter.split(np.zeros(len(target)), target, groups)):
        folds[val_idx] = fold
    return folds


results: list[dict] = []


def record(name: str, oof_pred, holdout_pred, score_fn=macro_f1) -> None:
    """Catat skor OOF dan holdout. Holdout hanya untuk laporan, bukan untuk memilih model."""
    results.append(
        {
            "model": name,
            "oof": round(score_fn(y, oof_pred), 4),
            "oof_ci95": bootstrap_interval(y, oof_pred, score_fn),
            "holdout": round(score_fn(y_holdout, holdout_pred), 4),
        }
    )
    print(results[-1])

**Training transformer.** Satu fungsi untuk klasifikasi (`problem="ce"`), multi-label (`"bce"`),
dan regresi (`"mse"`). Loop-nya ditulis sendiri supaya mudah dibaca dan outputnya bersih.

In [7]:
from torch.utils.data import DataLoader, TensorDataset  # noqa: E402
from transformers import (  # noqa: E402
    AutoModelForSequenceClassification,
    AutoTokenizer as _SeqTokenizer,
    get_linear_schedule_with_warmup,
)


def _encode(tokenizer, texts, pairs, max_len):
    args = (list(texts), list(pairs)) if pairs is not None else (list(texts),)
    tokens = tokenizer(
        *args, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt"
    )
    return tokens["input_ids"], tokens["attention_mask"]


def _loss_fn(problem, weights):
    if problem == "ce":
        return torch.nn.CrossEntropyLoss(weight=weights)
    # untuk multi-label, class_weights berperan sebagai pos_weight per label
    return (
        torch.nn.BCEWithLogitsLoss(pos_weight=weights) if problem == "bce" else torch.nn.MSELoss()
    )


def _to_output(logits, problem):
    if problem == "ce":
        return torch.softmax(logits.float(), dim=1)
    if problem == "bce":
        return torch.sigmoid(logits.float())
    return logits.float().squeeze(-1)


@torch.inference_mode()
def _predict(model, ids, mask, problem, batch_size):
    model.eval()
    chunks = []
    for start in range(0, len(ids), batch_size):
        batch_ids = ids[start : start + batch_size].to(DEVICE)
        batch_mask = mask[start : start + batch_size].to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            logits = model(input_ids=batch_ids, attention_mask=batch_mask).logits
        chunks.append(_to_output(logits, problem).cpu())
    return torch.cat(chunks).numpy()


def train_predict(
    model_name,
    train_texts,
    train_y,
    predict_sets,
    problem="ce",
    num_labels=None,
    class_weights=None,
    train_pairs=None,
    epochs=3,
    lr=2e-5,
    batch_size=32,
    max_len=128,
    seed=42,
):
    """Latih satu model, lalu kembalikan prediksi untuk tiap set di predict_sets.

    predict_sets: daftar teks, atau daftar pasangan (teks, teks_pair) untuk tugas dua teks.
    """
    seed_everything(seed)
    tokenizer = _SeqTokenizer.from_pretrained(MODEL_DIR / model_name)
    num_labels = num_labels or (int(np.max(train_y)) + 1 if problem == "ce" else 1)
    if problem == "bce":
        num_labels = np.asarray(train_y).shape[1]
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR / model_name, num_labels=num_labels, ignore_mismatched_sizes=True
    ).to(DEVICE)

    ids, mask = _encode(tokenizer, train_texts, train_pairs, max_len)
    target = torch.tensor(np.asarray(train_y), dtype=torch.long if problem == "ce" else torch.float)
    if problem == "mse":
        target = target.view(-1, 1)
    loader = DataLoader(
        TensorDataset(ids, mask, target),
        batch_size=batch_size,
        shuffle=True,
        drop_last=len(ids) > batch_size,
    )
    weights = (
        None if class_weights is None else torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
    )
    loss_fn = _loss_fn(problem, weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = max(1, epochs * len(loader))
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

    model.train()
    for _ in range(epochs):
        for batch_ids, batch_mask, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                logits = model(
                    input_ids=batch_ids.to(DEVICE), attention_mask=batch_mask.to(DEVICE)
                ).logits
                loss = loss_fn(logits, batch_y.to(DEVICE))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

    outputs = []
    for item in predict_sets:
        texts, pairs = item if isinstance(item, tuple) else (item, None)
        pred_ids, pred_mask = _encode(tokenizer, texts, pairs, max_len)
        outputs.append(_predict(model, pred_ids, pred_mask, problem, batch_size * 2))
    del model, optimizer, scaler
    torch.cuda.empty_cache()
    return outputs

**Embedding teks.** Kalau pretrained boleh dipakai, teks diubah dengan multilingual-e5. Kalau
tidak boleh, `USE_PRETRAINED=False` memakai TF-IDF + SVD sebagai gantinya.

In [8]:
from sklearn.decomposition import TruncatedSVD  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer as _Tfidf  # noqa: E402
from sklearn.preprocessing import normalize as _l2  # noqa: E402
from transformers import AutoModel, AutoTokenizer  # noqa: E402


def tfidf_svd_embed(texts: list[str], dim: int = 256) -> np.ndarray:
    """Embedding tanpa pretrained: TF-IDF kata + SVD."""
    matrix = _Tfidf(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(texts)
    dim = min(dim, matrix.shape[1] - 1, matrix.shape[0] - 1)
    return _l2(TruncatedSVD(dim, random_state=SEED).fit_transform(matrix))


@torch.inference_mode()
def e5_embed(texts: list[str], prefix: str = "query: ", batch_size: int = 64) -> np.ndarray:
    """Mean pooling multilingual-e5, hasil dinormalisasi L2."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR / "e5")
    model = AutoModel.from_pretrained(MODEL_DIR / "e5").to(DEVICE).eval().half()
    order = np.argsort([len(t) for t in texts])
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = [prefix + texts[i] for i in order[start : start + batch_size]]
        tokens = tokenizer(
            batch, padding=True, truncation=True, max_length=128, return_tensors="pt"
        ).to(DEVICE)
        hidden = model(**tokens).last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        chunks.append(torch.nn.functional.normalize(pooled, dim=1).float().cpu().numpy())
    del model
    torch.cuda.empty_cache()
    vectors = np.empty((len(texts), chunks[0].shape[1]), dtype=np.float32)
    vectors[order] = np.vstack(chunks)
    return vectors


def embed_texts(texts: list[str]) -> np.ndarray:
    """Pilih embedding sesuai USE_PRETRAINED."""
    return e5_embed(texts) if USE_PRETRAINED else tfidf_svd_embed(texts)

**Seq2seq.** `generate_texts` menghasilkan teks dari model pretrained. `finetune_seq2seq` melatih model
itu pada data lu sendiri. Bobot akhir tidak disimpan, supaya folder hasil tetap kecil.

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as SeqTokenizer  # noqa: E402


def load_seq2seq(name: str):
    tokenizer = SeqTokenizer.from_pretrained(MODEL_DIR / name)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR / name).to(DEVICE)
    return tokenizer, model


@torch.inference_mode()
def generate_texts(
    model, tokenizer, texts, max_input=256, max_new=64, beams=4, batch_size=32, prefix=""
):
    """Hasilkan teks, urut dari yang terpanjang supaya padding lebih hemat."""
    model.eval()
    order = np.argsort([-len(t) for t in texts])
    outputs = [""] * len(texts)
    for start in range(0, len(texts), batch_size):
        picks = order[start : start + batch_size]
        tokens = tokenizer(
            [prefix + texts[i] for i in picks],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input,
        ).to(DEVICE)
        generated = model.generate(
            **tokens, max_new_tokens=max_new, num_beams=beams, no_repeat_ngram_size=3
        )
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        for i, text in zip(picks, decoded):
            outputs[i] = text.strip()
    return outputs


def finetune_seq2seq(
    model,
    tokenizer,
    sources,
    targets,
    epochs,
    seed,
    lr=3e-5,
    batch_size=16,
    max_input=256,
    max_target=96,
    prefix="",
):
    """Latih model seq2seq dengan loss bawaan."""
    seed_everything(seed)
    source_tokens = tokenizer(
        [prefix + s for s in sources],
        truncation=True,
        max_length=max_input,
        padding="max_length",
        return_tensors="pt",
    )
    target_ids = tokenizer(
        text_target=list(targets),
        truncation=True,
        max_length=max_target,
        padding="max_length",
        return_tensors="pt",
    )["input_ids"]
    target_ids[target_ids == tokenizer.pad_token_id] = -100
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    generator = np.random.default_rng(seed)
    model.train()
    for _ in range(epochs):
        order = generator.permutation(len(sources))
        for start in range(0, len(order), batch_size):
            pick = torch.as_tensor(order[start : start + batch_size])
            optimizer.zero_grad(set_to_none=True)
            loss = model(
                input_ids=source_tokens["input_ids"][pick].to(DEVICE),
                attention_mask=source_tokens["attention_mask"][pick].to(DEVICE),
                labels=target_ids[pick].to(DEVICE),
            ).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return model

## 4. EDA

Muat data. Test resmi dipakai sebagai holdout.

In [10]:
def read_smsa(split: str) -> pd.DataFrame:
    path = EXTRA_DATA_DIR / "indonlu/dataset/smsa_doc-sentiment-prosa" / f"{split}_preprocess.tsv"
    return pd.read_csv(path, sep="\t", header=None, names=["text", "label"])


pool = read_smsa("train")
train = (
    pool.groupby("label", group_keys=False)
    .sample(frac=N_LABELED / len(pool), random_state=SEED)
    .reset_index(drop=True)
)
holdout = read_smsa("test")
print(f"latih={len(train)} | holdout={len(holdout)}")
print("proporsi kelas:", train["label"].value_counts(normalize=True).round(3).to_dict())
text_profile(train["text"])

latih=500 | holdout=500
proporsi kelas: {'positive': 0.584, 'negative': 0.312, 'neutral': 0.104}


baris                500.0
kosong                 0.0
duplikat_persis_%      0.0
kata_median           28.0
kata_p95              73.0
kata_maks             83.0
ada_url_%              0.0
ada_emoji_%            0.0
dtype: float64

**Temuan dan keputusan**

| Temuan | Keputusan |
|---|---|
| Label hanya 500 baris | Augmentasi hanya di data latih dalam fold |
| Netral 10,4% | Semua varian memakai fold yang sama |
| Back translation bisa mengubah makna | Kemiripan diukur (median 0,934), dan contoh dibaca manual |

## 5. Preprocessing

Pembersihan ringan. **Aturan utama: augmentasi hanya untuk data latih di dalam fold.** Kalau dilakukan sebelum
pembagian fold, salinan teks bocor ke validasi dan skornya palsu.

In [11]:
class_to_id = {name: i for i, name in enumerate(CLASSES)}
train["clean"] = train["text"].map(clean_text)
holdout["clean"] = holdout["text"].map(clean_text)
train["group"] = near_duplicate_groups(train["clean"].tolist())
y = train["label"].map(class_to_id).to_numpy()
y_holdout = holdout["label"].map(class_to_id).to_numpy()

**Augmentasi tanpa model.** Hapus kata acak, tukar kata bertetangga, dan ganti dengan bentuk informal dari kamus.

In [12]:
slang = load_slang_dict()
formal_to_slang: dict[str, str] = {}
for informal, formal in slang.items():
    if " " not in formal and len(informal) < len(formal):
        formal_to_slang.setdefault(formal, informal)


def word_dropout(text: str, rng, share: float = 0.1) -> str:
    words = text.split()
    kept = [w for w in words if rng.random() > share]
    return " ".join(kept) if len(kept) >= 3 else text


def word_swap(text: str, rng, n: int = 2) -> str:
    words = text.split()
    for _ in range(n):
        if len(words) > 2:
            i = rng.integers(0, len(words) - 1)
            words[i], words[i + 1] = words[i + 1], words[i]
    return " ".join(words)


def to_informal(text: str, rng, share: float = 0.3) -> str:
    return " ".join(
        formal_to_slang[w] if w in formal_to_slang and rng.random() < share else w
        for w in text.split()
    )

**Back-translation.** Indonesia ke Inggris lalu kembali. Hasilnya dihitung sekali untuk semua teks latih, tapi tetap
hanya masuk ke fold latih.

In [13]:
back_translated = None
if USE_PRETRAINED:
    with stage("back_translation"):
        tok_forward, model_forward = load_seq2seq("marian_id_en")
        english = generate_texts(
            model_forward, tok_forward, train["clean"].tolist(), max_input=128, max_new=128, beams=1
        )
        del model_forward
        tok_back, model_back = load_seq2seq("marian_en_id")
        back_translated = generate_texts(
            model_back, tok_back, english, max_input=128, max_new=128, beams=1
        )
        del model_back
        torch.cuda.empty_cache()
    print("asli:", train["clean"][0][:90])
    print("hasil:", back_translated[0][:90])

[waktu] back_translation: 166 detik
asli: kita akhir pekan ke sana . sesak dan antre . beli voucher makan dulu . dari beberapa makan
hasil: Kita akan pergi akhir pekan, mengumpulkan dan menunggu, membeli voucher pertama.


## 6. Validasi

Skema `StratifiedGroupKFold`, metrik macro-F1. Semua varian memakai fold yang sama, supaya perbandingannya adil.
Augmentasi diterima kalau interval perbedaannya jelas di atas nol, bukan hanya rata-ratanya naik.

In [14]:
folds = make_folds(y, train["group"], SETTINGS["folds"])
print("baris per fold:", np.bincount(folds).tolist())


def paired_gain(base_pred: np.ndarray, new_pred: np.ndarray, n_boot: int = 500) -> dict[str, float]:
    rng = np.random.default_rng(SEED)
    gains = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(y), len(y))
        gains.append(macro_f1(y[pick], new_pred[pick]) - macro_f1(y[pick], base_pred[pick]))
    return {
        "kenaikan": round(float(np.mean(gains)), 4),
        "p_lebih_baik": round(float(np.mean(np.array(gains) > 0)), 3),
    }

baris per fold: [101, 99, 100, 100, 100]


## 7. Baseline

Tanpa augmentasi. Kalau `USE_PRETRAINED=False`, classifier memakai TF-IDF + regresi logistik.

In [15]:
def fit_predict(texts, labels, predict_sets, seed):
    if USE_PRETRAINED:
        weights = len(labels) / (len(CLASSES) * np.bincount(labels, minlength=len(CLASSES)))
        return train_predict(
            "indobertweet",
            list(texts),
            labels,
            predict_sets,
            num_labels=len(CLASSES),
            class_weights=weights,
            epochs=SETTINGS["epochs"],
            seed=seed,
        )
    vector = _Tfidf(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    model = LogisticRegression(C=3.0, max_iter=2000, class_weight="balanced").fit(
        vector.fit_transform(texts), labels
    )
    return [model.predict_proba(vector.transform(s)) for s in predict_sets]


def run_cv(augment_fn=None) -> tuple[np.ndarray, np.ndarray]:
    """Cross-validation. augment_fn menerima indeks latih dan mengembalikan teks tambahan."""
    oof = np.zeros((len(y), len(CLASSES)))
    hold = np.zeros((len(y_holdout), len(CLASSES)))
    n_runs = SETTINGS["folds"] * len(SETTINGS["seeds"])
    for seed in SETTINGS["seeds"]:
        for fold in range(SETTINGS["folds"]):
            fit_idx, val_idx = np.where(folds != fold)[0], np.where(folds == fold)[0]
            texts = train["clean"].iloc[fit_idx].tolist()
            labels = y[fit_idx]
            if augment_fn is not None:
                texts = texts + augment_fn(fit_idx)
                labels = np.concatenate([labels, y[fit_idx]])
            pred_val, pred_hold = fit_predict(
                texts,
                labels,
                [train["clean"].iloc[val_idx].tolist(), holdout["clean"].tolist()],
                seed + fold,
            )
            oof[val_idx] += pred_val / len(SETTINGS["seeds"])
            hold += pred_hold / n_runs
    return oof, hold


with stage("tanpa_augmentasi"):
    oof_base, hold_base = run_cv()
record("tanpa_augmentasi", oof_base.argmax(1), hold_base.argmax(1))

[waktu] tanpa_augmentasi: 183 detik


{'model': 'tanpa_augmentasi', 'oof': 0.8336, 'oof_ci95': '0.790-0.872', 'holdout': 0.6265}


## 8. Model Utama

Latih ulang dengan tiap augmentasi pada fold yang sama.

In [16]:
def word_augment(fn):
    def augment(index: np.ndarray) -> list[str]:
        rng = np.random.default_rng(SEED)
        return [fn(train["clean"].iloc[i], rng) for i in index]

    return augment


AUGMENTERS = {
    "hapus_kata": word_augment(word_dropout),
    "tukar_kata": word_augment(word_swap),
    "slang": word_augment(to_informal),
}
if back_translated is not None:
    AUGMENTERS["back_translation"] = lambda index: [back_translated[i] for i in index]

outcomes = {"tanpa_augmentasi": (oof_base, hold_base)}
for name, fn in AUGMENTERS.items():
    with stage(name):
        outcomes[name] = run_cv(fn)
    record(name, outcomes[name][0].argmax(1), outcomes[name][1].argmax(1))

[waktu] hapus_kata: 338 detik


{'model': 'hapus_kata', 'oof': 0.8406, 'oof_ci95': '0.793-0.876', 'holdout': 0.7916}


[waktu] tukar_kata: 339 detik


{'model': 'tukar_kata', 'oof': 0.8502, 'oof_ci95': '0.805-0.886', 'holdout': 0.7946}


[waktu] slang: 339 detik


{'model': 'slang', 'oof': 0.8518, 'oof_ci95': '0.810-0.885', 'holdout': 0.8005}


[waktu] back_translation: 337 detik


{'model': 'back_translation', 'oof': 0.8454, 'oof_ci95': '0.801-0.885', 'holdout': 0.7577}


## 9. Post-processing

Pilih varian dengan peluang kenaikan terbaik di OOF. Kalau tidak ada yang melewati 0,9, tetap tanpa augmentasi.

In [17]:
gains = {
    n: paired_gain(oof_base.argmax(1), o[0].argmax(1))
    for n, o in outcomes.items()
    if n != "tanpa_augmentasi"
}
gain_table = pd.DataFrame(gains).T.sort_values("p_lebih_baik", ascending=False)
best_name = gain_table.index[0] if gain_table["p_lebih_baik"].iloc[0] >= 0.9 else "tanpa_augmentasi"
print("dipilih:", best_name)
gain_table

dipilih: tanpa_augmentasi


,kenaikan,p_lebih_baik
slang,0.0173,0.884
tukar_kata,0.0162,0.858
back_translation,0.0118,0.812
hapus_kata,0.0063,0.698


## 10. Post-analysis

**Tabel perbandingan.**

In [18]:
result_table = pd.DataFrame(results)
result_table.to_csv(RESULT_DIR / "perbandingan.csv", index=False)
result_table

,model,oof,oof_ci95,holdout
0,tanpa_augmentasi,0.8336,0.790-0.872,0.6265
1,hapus_kata,0.8406,0.793-0.876,0.7916
2,tukar_kata,0.8502,0.805-0.886,0.7946
3,slang,0.8518,0.810-0.885,0.8005
4,back_translation,0.8454,0.801-0.885,0.7577


**Apakah makna dan label terjaga?** Kemiripan embedding teks asli dan hasil augmentasi. Kalau terlalu tinggi, teks hampir
sama dan tidak menambah informasi. Kalau terlalu rendah, label mungkin sudah berubah.

In [19]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(train), min(200, len(train)), replace=False)
originals = train["clean"].iloc[sample_idx].tolist()
variants = {
    "hapus_kata": [word_dropout(t, rng) for t in originals],
    "tukar_kata": [word_swap(t, rng) for t in originals],
    "slang": [to_informal(t, rng) for t in originals],
}
if back_translated is not None:
    variants["back_translation"] = [back_translated[i] for i in sample_idx]
base_vectors = embed_texts(originals)
similarity_rows = []
for name, texts in variants.items():
    changed = np.mean([a != b for a, b in zip(originals, texts)])
    similarity_rows.append(
        {
            "augmentasi": name,
            "kemiripan_median": round(
                float(np.median(np.sum(base_vectors * embed_texts(texts), axis=1))), 3
            ),
            "teks_berubah": round(float(changed), 3),
        }
    )
pd.DataFrame(similarity_rows)

,augmentasi,kemiripan_median,teks_berubah
0,hapus_kata,0.995,0.830
1,tukar_kata,0.999,0.880
2,slang,0.989,0.835
3,back_translation,0.934,0.980


**Contoh hasil,** untuk dicek manual apakah label masih benar.

In [20]:
pd.DataFrame(
    {
        "asli": [t[:70] for t in originals[:5]],
        **{k: [t[:70] for t in v[:5]] for k, v in variants.items()},
    }
)

,asli,hapus_kata,tukar_kata,slang,back_translation
0,"kenapa tidak rekomendasi ? 1 . pempek belum matang , tapi sudah disaji","kenapa tidak rekomendasi ? . pempek belum matang , tapi sudah disajika","kenapa tidak rekomendasi ? 1 . pempek belum matang , tapi sudah disaji","kenapa tidak rekomendasi ? 1 . pempek blm matang , tapi sudah disajika","Kenapa bukan rekomendasi? 1.pek belum dimasak, tapi sudah disajikan 2."
1,mi di sini terkenal enak . sayang porsi nya sedikit sekali . bakso gor,mi sini terkenal enak . sayang nya sedikit sekali . goreng juga enak .,di mi sini terkenal . enak sayang porsi nya sedikit sekali . bakso gor,mi di sini trknal enk . sayang porsi ny sdkit sekali . bakso goreng ny,Berikut adalah lezat terkenal.
2,lokasi daerah maribaya lembang bandung . agak jauh dari keramaian kota,lokasi daerah maribaya lembang bandung . agak jauh dari keramaian kota,lokasi daerah maribaya lembang bandung . agak jauh dari keramaian kota,lokasi daerah maribaya lembang bandung . agak jauh dr keramaian kota d,Lokasi daerah maribaka yang mengapung itu agak jauh dari kerumunan kot
3,saya membenci suporter persija,saya membenci suporter persija,saya membenci suporter persija,saya membenci suporter persija,Aku benci pendukung pertunjukan.
4,"gerindra , partai pki zaman sekarang . pendana mca atau saracen . peme","gerindra , partai pki zaman sekarang . pendana mca atau saracen . peme","gerindra , partai pki sekarang zaman . pendana atau mca saracen . peme","gerindra , partai pki zaman sekarang . pendana mca atau saracen . peme","Geridra, pesta pki hari ini."


## 11. Keluaran

File untuk dikumpulkan dan ringkasan run.

In [21]:
final_hold = outcomes[best_name][1].argmax(1)
pd.DataFrame({"id": holdout.index, "label": [CLASSES[i] for i in final_hold]}).to_csv(
    RESULT_DIR / "submission.csv", index=False
)
run_summary(
    "augmentasi",
    len(train),
    {
        "dipilih": best_name,
        **{r["model"]: {"oof": r["oof"], "holdout": r["holdout"]} for r in results},
    },
    needs_gpu=USE_PRETRAINED,
)

RINGKASAN_RUN task=augmentasi mode=penuh n_rows=500 total_detik=1757 gpu=True
{
 "dipilih": "tanpa_augmentasi",
 "tanpa_augmentasi": {
  "oof": 0.8336,
  "holdout": 0.6265
 },
 "hapus_kata": {
  "oof": 0.8406,
  "holdout": 0.7916
 },
 "tukar_kata": {
  "oof": 0.8502,
  "holdout": 0.7946
 },
 "slang": {
  "oof": 0.8518,
  "holdout": 0.8005
 },
 "back_translation": {
  "oof": 0.8454,
  "holdout": 0.7577
 }
}


**Poin untuk presentasi**

- Tidak ada augmentasi yang lolos syarat peluang 0,9. Terbaik: slang, kenaikan OOF 0,0173 dengan peluang 0,884.
- Semua augmentasi menaikkan holdout dari 0,6265 ke 0,76-0,80. Augmentasi juga menggandakan jumlah langkah latih, jadi kenaikan ini belum tentu karena isi augmentasinya. Perlu pembanding epoch lebih banyak tanpa augmentasi.
- Aturan memilih dari OOF menolak semua augmentasi, padahal holdout-nya naik. Saat data sangat sedikit dan bergeser, OOF bisa kurang peka.
- Back translation paling mahal, dan holdout-nya terendah di antara augmentasi (0,7577).